# Constitutional AI / RLAIF — Colab Quickstart

Runs the full pipeline end to end on a free T4 GPU:

1. Generate the prompt dataset (benign / boundary / adversarial)
2. Sample 2 responses per prompt from the base model and have an AI judge build preference pairs from a written constitution (this is the RLAIF step)
3. Fine-tune the base model with LoRA + DPO on those preference pairs
4. Evaluate base vs. tuned model on held-out prompts
5. Run the red-teaming loop (adversarial robustness across rounds)
6. Launch the interactive Gradio demo

**Before running:** upload this whole `constitutional-ai-rl/` project folder to Colab
(drag it into the Files panel on the left, or clone it from your own GitHub repo with
`!git clone <your-repo-url>`), then set `PROJECT_DIR` in the next cell to wherever it landed.

**Runtime:** Runtime > Change runtime type > T4 GPU, before running any cells.

In [ ]:
PROJECT_DIR = "/content/constitutional-ai-rl"  # change if you uploaded/cloned elsewhere

import os
assert os.path.isdir(PROJECT_DIR), f"{PROJECT_DIR} not found — upload/clone the project first"
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU detected — check Runtime > Change runtime type")

## (Optional) Use Claude as the judge instead of the local fallback model

The judge defaults to a small local model (Qwen2.5-1.5B-Instruct), which is free but noisy.
For meaningfully better judge quality, set an API key below — every later script auto-detects it,
no other changes needed.

In [ ]:
import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # uncomment and fill in to use Claude as the judge

## Step 1: generate the prompt dataset

In [ ]:
!python src/generate_prompts.py

## Step 2: sample responses + judge them into preference pairs (RLAIF)

Drop `--limit` (or raise it) for a larger run once you've confirmed everything works.

In [ ]:
!python src/build_preference_dataset.py --limit 40

## Step 3: fine-tune with LoRA + DPO

In [ ]:
!python src/train_dpo.py --epochs 3

## Step 4: evaluate base vs. tuned model on held-out prompts

In [ ]:
!python src/evaluate.py

In [ ]:
from IPython.display import Image, display
display(Image(filename="results/plots/violations_by_rule.png"))

## Step 5: red-teaming loop (adversarial robustness across rounds)

Each round: attack the current model with adversarial prompts, retrain on the failures, repeat.
Watch `jailbreak_success_rate` drop round over round.

In [ ]:
!python src/red_team.py --rounds 3

In [ ]:
display(Image(filename="results/plots/red_team_success_rate.png"))

## Step 6: launch the interactive demo

Colab has no local browser to hit `localhost`, so launch with `share=True` to get a public Gradio link.

In [ ]:
import sys
sys.path.insert(0, "app")
from demo import build_app

app = build_app()
app.launch(share=True)